In [3]:
import pandas as pd
import numpy as np
import os
import time

%run milp_engine.ipynb

In [ ]:
# ==========================================================
# 1. EXPERIMENT CONFIGURATION
# ==========================================================

MODEL_NAME = "RF"

INPUT_FILES = {

    "LightGBM":
        "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",

    "RF":
        "Prescriptive/RF_prescriptive_optimizer_inputs.csv",

    "XGBoost":
        "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv"
}

INPUT_FILE = INPUT_FILES[MODEL_NAME]

N_MONTE_CARLO = 100

COHORT_SIZE = 12

LAMBDA_LIST = np.round(

    np.linspace(
        0.0,
        1.0,
        11
    ),

    2

)

OUTPUT_DIR = f"{MODEL_NAME}_Results"

os.makedirs(

    OUTPUT_DIR,

    exist_ok=True

)

# ==========================================================
# 2. LOAD DATA
# ==========================================================

print(

    f"\nLoading dataset: {INPUT_FILE}"

)

df = pd.read_csv(

    INPUT_FILE,

    encoding="utf-8"

)

print(

    f"Total cases available: {len(df)}"

)

# ==========================================================
# 3. MONTE CARLO LOOP
# ==========================================================

all_results = []

global_start = time.time()

for sim in range(N_MONTE_CARLO):

    print(

        f"\n========================="

    )

    print(

        f"Monte Carlo Run {sim + 1}/{N_MONTE_CARLO}"

    )

    print(

        f"========================="

    )

    # --------------------------------------------------
    # Random cohort draw
    # --------------------------------------------------

    cohort = df.sample(

        n=COHORT_SIZE,

        random_state=sim

    ).reset_index(

        drop=True

    )

    # --------------------------------------------------
    # Lambda loop
    # --------------------------------------------------

    for lam in LAMBDA_LIST:

        print(

            f"Lambda = {lam:.1f}"

        )

        result = solve_or_milp(

            cohort,

            lam

        )

        result["Simulation"] = sim

        result["Lambda"] = lam

        result["Model"] = MODEL_NAME

        all_results.append(

            result

        )

# ==========================================================
# 4. SAVE RAW RESULTS
# ==========================================================

raw_df = pd.DataFrame(

    all_results

)

raw_file = os.path.join(

    OUTPUT_DIR,

    "monte_carlo_raw.csv"

)

raw_df.to_csv(

    raw_file,

    index=False

)

print(

    f"\nRaw Monte Carlo results saved to:\n{raw_file}"

)

# ==========================================================
# 5. SUMMARY STATISTICS
# ==========================================================

summary = raw_df.groupby(

    "Lambda"

).agg({

    "Objective":

        ["mean", "std"],

    "Overtime":

        ["mean", "std"],

    "Idle":

        ["mean", "std"],

    "Makespan":

        ["mean", "std"],

    "Sched_Util":

        ["mean", "std"],

    "Cap_Expansion":

        ["mean", "std"],

    "Solve_Time":

        ["mean", "std"]

})

summary.columns = [

    "_".join(col)

    for col in summary.columns

]

summary = summary.reset_index()

# ==========================================================
# 6. 95% CONFIDENCE INTERVALS
# ==========================================================

N = N_MONTE_CARLO

for metric in [

    "Objective",

    "Overtime",

    "Makespan",

    "Cap_Expansion"

]:

    summary[

        metric + "_CI95"

    ] = (

        1.96

        *

        summary[

            metric + "_std"

        ]

        /

        np.sqrt(

            N

        )

    )

# ==========================================================
# 7. SAVE SUMMARY
# ==========================================================

summary_file = os.path.join(

    OUTPUT_DIR,

    "monte_carlo_summary.csv"

)

summary.to_csv(

    summary_file,

    index=False

)

print(

    f"\nSummary saved to:\n{summary_file}"

)

# ==========================================================
# 8. FINAL REPORT
# ==========================================================

elapsed = time.time() - global_start

print("\n" + "=" * 90)

print(

    "MONTE CARLO EXPERIMENT COMPLETE"

)

print("=" * 90)

print(

    summary.to_string(

        index=False

    )

)

print("=" * 90)

print(

    f"\nTotal runtime: {elapsed:.2f} sec"

)

print(

    f"Average runtime per scenario: "

    f"{elapsed / (N_MONTE_CARLO * len(LAMBDA_LIST)):.2f} sec"

)


Loading dataset: Prescriptive/RF_prescriptive_optimizer_inputs.csv
Total cases available: 16140

Monte Carlo Run 1/100
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Monte Carlo Run 2/100
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Monte Carlo Run 3/100
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Monte Carlo Run 4/100
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Monte Carlo Run 5/100
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Monte Carlo Run 6/100
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.